# Chapter 24 — Stochastic Processes
*Tier 3: All Tracks*

## 🎯 Learning Objectives
- Understand key stochastic processes: Brownian motion, random walks, Poisson processes
- Simulate and analyse paths of continuous-time processes
- Apply to finance, engineering, and diffusion problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Key Stochastic Processes

| Process | Definition | Key property |
|---------|-----------|-------------|
| Random Walk | $S_n = \sum_{i=1}^n X_i$, $X_i \in \{\pm 1\}$ | Discrete, integer-valued |
| Brownian Motion | $W(t) = \lim$ of scaled random walk | Continuous paths, $W(t) \sim N(0,t)$ |
| Geometric BM | $S(t) = S_0 e^{(\mu-\sigma^2/2)t + \sigma W(t)}$ | Always positive, used in finance |
| Ornstein-Uhlenbeck | $dX = \theta(\mu - X)dt + \sigma dW$ | Mean-reverting |

In [ ]:
# Standard Brownian motion via random walk limit
dt = 0.01
T  = 10
t  = np.arange(0, T+dt, dt)
n_paths = 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for _ in range(n_paths):
    increments = rng.normal(0, np.sqrt(dt), len(t)-1)
    W = np.concatenate([[0], np.cumsum(increments)])
    axes[0].plot(t, W, alpha=0.7, lw=0.8)
axes[0].set_title("Standard Brownian Motion")
axes[0].set_xlabel("t"); axes[0].set_ylabel("W(t)")

# Distribution at various times
for time_pt in [1, 4, 9]:
    samples = rng.normal(0, np.sqrt(time_pt), 5000)
    axes[1].hist(samples, bins=50, density=True, alpha=0.5, label=f"t={time_pt}")
    x_r = np.linspace(-10, 10, 300)
    axes[1].plot(x_r, stats.norm.pdf(x_r, 0, np.sqrt(time_pt)), lw=2)
axes[1].set_title("W(t) ~ N(0,t) at Different Times"); axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Geometric Brownian Motion (GBM)

$$dS = \mu S\, dt + \sigma S\, dW$$

Solution (Itô's lemma):
$$S(t) = S_0 \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)t + \sigma W(t)\right)$$

Properties:
- $E[S(t)] = S_0 e^{\mu t}$
- $\text{Var}[S(t)] = S_0^2 e^{2\mu t}(e^{\sigma^2 t} - 1)$

In [ ]:
# GBM simulation (stock price)
S0 = 100; mu = 0.08; sigma = 0.20
T_years = 2; n_steps = 504  # ~252 trading days per year
dt = T_years / n_steps
t_yr = np.linspace(0, T_years, n_steps+1)
n_paths = 500

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
final_prices = []
for _ in range(n_paths):
    Z = rng.normal(0, 1, n_steps)
    increments = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    log_S = np.log(S0) + np.concatenate([[0], np.cumsum(increments)])
    S = np.exp(log_S)
    final_prices.append(S[-1])
    if _ < 50:
        axes[0].plot(t_yr, S, alpha=0.15, lw=0.6, color="steelblue")
E_S = S0 * np.exp(mu * T_years)
axes[0].plot(t_yr, S0*np.exp(mu*t_yr), "r-", lw=2, label=f"E[S]={E_S:.0f}")
axes[0].set_title(f"GBM — {n_paths} paths, μ={mu}, σ={sigma}")
axes[0].set_xlabel("Years"); axes[0].set_ylabel("Price")
axes[0].legend()

axes[1].hist(final_prices, bins=60, density=True, alpha=0.7, edgecolor="none")
axes[1].axvline(np.mean(final_prices), color="red", lw=2, label=f"Mean={np.mean(final_prices):.0f}")
axes[1].set_title(f"Distribution of S(T={T_years})")
axes[1].legend()
plt.tight_layout(); plt.show()

# Option pricing (European call)
K = 100  # strike
call_payoffs = np.maximum(np.array(final_prices) - K, 0)
call_price = np.exp(-0.04 * T_years) * np.mean(call_payoffs)
print(f"Monte Carlo call price (K=100, r=4%): ${call_price:.2f}")

## 3. Ornstein-Uhlenbeck (Mean-Reverting) Process

In [ ]:
# Ornstein-Uhlenbeck: interest rate model
theta = 2.0   # mean reversion speed
mu_ou = 0.05  # long-run mean
sigma_ou = 0.02
X0 = 0.10
dt = 1/252; T_ou = 5; n_ou = int(T_ou/dt)
t_ou = np.linspace(0, T_ou, n_ou+1)

n_paths_ou = 5
plt.figure(figsize=(10, 5))
for _ in range(n_paths_ou):
    X = np.zeros(n_ou+1); X[0] = X0
    for k in range(n_ou):
        X[k+1] = (X[k] + theta*(mu_ou - X[k])*dt +
                  sigma_ou*np.sqrt(dt)*rng.standard_normal())
    plt.plot(t_ou, X*100, alpha=0.8, lw=1)
plt.axhline(mu_ou*100, color="red", lw=2, linestyle="--", label=f"Long-run mean={mu_ou*100:.1f}%")
plt.xlabel("Years"); plt.ylabel("Interest rate (%)")
plt.title("Ornstein-Uhlenbeck Process (Vasicek Rate Model)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Practice: First passage time
# How long until BM first hits level a?
a = 2.0
n_trials = 10_000
dt_fine = 0.001
first_passage = []
for _ in range(n_trials):
    W = 0
    t = 0
    for _ in range(100_000):
        W += rng.normal(0, np.sqrt(dt_fine))
        t += dt_fine
        if W >= a:
            first_passage.append(t)
            break

print(f"Simulated E[T_a={a}] = {np.mean(first_passage):.4f}")
print(f"Analytic: the first passage time has an inverse Gaussian distribution")
print(f"  (The distribution is heavy-tailed — mean may not converge for large a)")